# LNDL Test Cases

Comprehensive tests for `branch.operate(lndl=True)` covering various schema shapes.

**LNDL** (Language Network Directive Language) lets the model declare values via `<lvar>` tags and tool calls via `<lact>` tags, then commits which aliases are final via `OUT{}`.

Each case lists:
- **Schema**: the Pydantic model(s) defining the response
- **Expected**: the shape of the returned object
- **What we test**: the LNDL pattern under examination

In [ ]:
from pydantic import BaseModel, Field
from lionagi import Branch, iModel

MODEL = "openai/gpt-5.4-mini"


def multiply(number1: float, number2: float) -> float:
    """Multiply two numbers."""
    return number1 * number2


def summarize(text: str) -> str:
    """Return a one-line summary of the given text."""
    return f"Summary: {text[:50]}..."


def search_web(query: str, limit: int = 5) -> list[str]:
    """Search the web (stub). Returns mock results."""
    return [f"Result {i+1} for {query!r}" for i in range(limit)]


def fresh_branch(system: str = "You produce clean LNDL output.", tools=None):
    return Branch(
        system=system,
        tools=tools,
        chat_model=iModel(model=MODEL),
    )


def show(label, out):
    print(f"=== {label} ===")
    print(f"type: {type(out).__name__}")
    print(f"value: {out!r}")
    print()

## Case 1 — Scalar specs filled by tool calls

**Schema**: two `int` fields.
**Pattern**: each spec filled by an `<lact>` tag.
**Expected**: `AnswerResponse(q1=int, q2=int, action_requests=[...], action_responses=[...])`.

In [ ]:
class Answer1(BaseModel):
    q1: int
    q2: int


branch = fresh_branch(tools=multiply)
out = await branch.operate(
    instruction="Compute 3*4 for q1 and 3*2 for q2 using the multiply tool.",
    actions=True,
    lndl=True,
    response_format=Answer1,
)
show("Case 1", out)
print(branch.messages[-1].rendered if hasattr(branch.messages[-1], 'rendered') else '')
print("Raw assistant:\n", branch.messages[2].content.assistant_response if len(branch.messages) > 2 else 'n/a')

## Case 2 — Pure structured extraction (no tools)

**Schema**: `name: str, age: int`.
**Pattern**: each spec filled by an `<lvar>` tag.
**Expected**: `Person(name="Alice", age=30)`.

In [ ]:
class Person(BaseModel):
    name: str
    age: int


branch = fresh_branch()
out = await branch.operate(
    instruction="Extract: 'Alice is thirty years old.'",
    lndl=True,
    response_format=Person,
)
show("Case 2", out)

## Case 3 — Single nested-model field with mixed `<lvar>` + `<lact>`

**Schema**: `report: Report(title, summary)`.
**Pattern**: title from `<lvar>`, summary from `<lact summarize>`.
**Expected**: `ContainerResponse(report=Report(title=..., summary=...), action_requests=[...], action_responses=[...])`.

In [ ]:
class Report(BaseModel):
    title: str
    summary: str

class Container3(BaseModel):
    report: Report


branch = fresh_branch(tools=summarize)
out = await branch.operate(
    instruction=(
        "Build a Report. Title should be 'AI Safety Analysis'. "
        "For the summary, call summarize with text='AI safety is a critical field studying alignment'."
    ),
    actions=True,
    lndl=True,
    response_format=Container3,
)
show("Case 3", out)
print("Raw assistant:\n", branch.messages[2].content.assistant_response if len(branch.messages) > 2 else 'n/a')

## Case 4 — List of strings

**Schema**: `findings: list[str]`.
**Pattern**: multiple `<lvar>` tags, OUT references multiple aliases.
**Expected**: `FindingsList(findings=["...", "...", "..."])`.

In [ ]:
class FindingsList(BaseModel):
    findings: list[str]


branch = fresh_branch()
out = await branch.operate(
    instruction="List three key benefits of unit testing as findings.",
    lndl=True,
    response_format=FindingsList,
)
show("Case 4", out)
print("Raw assistant:\n", branch.messages[2].content.assistant_response if len(branch.messages) > 2 else 'n/a')

## Case 5 — List of nested models

**Schema**: `items: list[Finding(name, score)]`.
**Pattern**: multiple Finding instances declared via `<lvar>` tags.
**Expected**: `Catalog(items=[Finding(...), Finding(...)])`.

In [ ]:
class Finding(BaseModel):
    name: str
    score: float

class Catalog(BaseModel):
    items: list[Finding]


branch = fresh_branch()
out = await branch.operate(
    instruction="Rate three Python web frameworks (django, flask, fastapi) on a 0-1 scale based on async support.",
    lndl=True,
    response_format=Catalog,
)
show("Case 5", out)
print("Raw assistant:\n", branch.messages[2].content.assistant_response if len(branch.messages) > 2 else 'n/a')

## Case 6 — Optional / nullable field

**Schema**: `required: str, optional: str | None = None`.
**Pattern**: omit optional field from OUT.
**Expected**: `OptDoc(required="...", optional=None)`.

In [ ]:
class OptDoc(BaseModel):
    required: str
    optional: str | None = None


branch = fresh_branch()
out = await branch.operate(
    instruction="Set required to 'hello'. Leave optional unset.",
    lndl=True,
    response_format=OptDoc,
)
show("Case 6", out)
print("Raw assistant:\n", branch.messages[2].content.assistant_response if len(branch.messages) > 2 else 'n/a')

## Case 7 — Reason + actions + LNDL

**Schema**: `Answer(q: int)` with `reason=True`.
**Pattern**: model adds Reason field via operative; LNDL renders the prompt without action fields.
**Expected**: response with `q`, `reason`, `action_requests`, `action_responses`.

In [ ]:
class Q(BaseModel):
    q: int


branch = fresh_branch(tools=multiply)
out = await branch.operate(
    instruction="What is 5 * 7? Use the multiply tool. Explain your reasoning briefly.",
    reason=True,
    actions=True,
    lndl=True,
    response_format=Q,
)
show("Case 7", out)
print("Raw assistant:\n", branch.messages[2].content.assistant_response if len(branch.messages) > 2 else 'n/a')

## Case 8 — Multiple top-level fields with mixed types

**Schema**: `report: Report, score: float, tags: list[str]`.
**Pattern**: nested model + scalar + list.
**Expected**: full validated model.

In [ ]:
class Container8(BaseModel):
    report: Report
    score: float
    tags: list[str]


branch = fresh_branch()
out = await branch.operate(
    instruction=(
        "Build a Container with: a Report (title='Climate Update', summary='Carbon levels rising'), "
        "score=0.85, and tags=['climate', 'science', 'urgent']."
    ),
    lndl=True,
    response_format=Container8,
)
show("Case 8", out)
print("Raw assistant:\n", branch.messages[2].content.assistant_response if len(branch.messages) > 2 else 'n/a')

## Case 9 — Dict field

**Schema**: `metrics: dict[str, float]`.
**Pattern**: free-form key-value extraction.
**Expected**: `Metrics(metrics={'precision': 0.9, 'recall': 0.85})`.

In [ ]:
class Metrics(BaseModel):
    metrics: dict[str, float]


branch = fresh_branch()
out = await branch.operate(
    instruction="Report metrics: precision=0.9, recall=0.85, f1=0.87.",
    lndl=True,
    response_format=Metrics,
)
show("Case 9", out)
print("Raw assistant:\n", branch.messages[2].content.assistant_response if len(branch.messages) > 2 else 'n/a')

## Case 10 — Union type field

**Schema**: `result: int | str` (model picks one).
**Pattern**: field can hold either type.
**Expected**: `Either(result=42)` or `Either(result='success')`.

In [ ]:
class Either(BaseModel):
    result: int | str


branch = fresh_branch()
out = await branch.operate(
    instruction="Set result to the integer 42.",
    lndl=True,
    response_format=Either,
)
show("Case 10", out)
print("Raw assistant:\n", branch.messages[2].content.assistant_response if len(branch.messages) > 2 else 'n/a')

## Summary

Run cases above and inspect both the parsed output and the raw assistant LNDL text. Mismatches between expected and actual fall into two buckets:

1. **Prompt engineering** — model produced incorrect LNDL syntax, picked wrong tags, or didn't follow OUT{} convention.
2. **Code logic** — parser/operate pipeline mishandled a valid LNDL response (e.g. dropped fields, failed validation, returned dict instead of model).